# Clustering-Based Course Recommender

This notebook groups users by genre-interest profiles and recommends popular unseen courses from the user's cluster. It follows the presentation flow: standardization, PCA, KMeans, and cluster-based recommendations.

## Dependency note

This notebook uses scikit-learn for `StandardScaler`, `PCA`, and `KMeans`. Install the requirements before running it.

In [1]:
from pathlib import Path
import urllib.request
import pandas as pd
import numpy as np

DATA_DIR = Path("datasets")
DATA_URLS = {
    "ratings.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/ratings.csv",
    "course_genre.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/course_genre.csv",
    "rs_content_test.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/rs_content_test.csv",
    "user_profile.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/user_profile.csv",
    "course_processed.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/course_processed.csv",
    "courses_bows.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/courses_bows.csv",
    "sim.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/sim.csv",
}

def ensure_dataset(filename):
    DATA_DIR.mkdir(exist_ok=True)
    path = DATA_DIR / filename
    if not path.exists():
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(DATA_URLS[filename], path)
    return path

def load_csv(filename, **kwargs):
    return pd.read_csv(ensure_dataset(filename), **kwargs)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

user_profile_df = load_csv("user_profile.csv")
test_users_df = load_csv("rs_content_test.csv")
ratings_df = load_csv("ratings.csv")

genre_cols = [col for col in user_profile_df.columns if col != "user"]
features = user_profile_df[genre_cols].to_numpy(dtype=float)
user_ids = user_profile_df["user"].to_numpy()

print("User profile data:", user_profile_df.shape)
print("Number of genre features:", len(genre_cols))

User profile data: (33901, 15)
Number of genre features: 14


## Standardization and PCA

PCA reduces the 14 genre dimensions while retaining at least 90% of the variance.

In [3]:
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

pca_full = PCA(random_state=42)
pca_full.fit(scaled_features)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
n_components = int(np.searchsorted(cumulative_variance, 0.90) + 1)

pca = PCA(n_components=n_components, random_state=42)
pca_features = pca.fit_transform(scaled_features)

print("PCA components selected:", n_components)
print(f"Variance retained: {pca.explained_variance_ratio_.sum():.3f}")

PCA components selected: 9
Variance retained: 0.927


## KMeans Clustering

The presentation uses the elbow-method idea and proceeds with a compact set of user clusters. The code below uses 5 clusters for a reproducible capstone run.

In [4]:
N_CLUSTERS = 5

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(pca_features)

clustered_users_df = pd.DataFrame({"user": user_ids, "cluster": cluster_labels})
display(clustered_users_df["cluster"].value_counts().sort_index().rename("user_count"))

cluster
0    16886
1     1951
2       20
3     4213
4    10831
Name: user_count, dtype: int64

## Cluster-Based Recommendations

For each test user, find the user's cluster and recommend popular courses taken by other users in that cluster that the target user has not taken.

In [5]:
ENROLLMENT_THRESHOLD = 10

ratings_with_cluster = ratings_df.merge(clustered_users_df, on="user", how="inner")
cluster_course_counts = (
    ratings_with_cluster.groupby(["cluster", "item"])
    .size()
    .rename("cluster_enrollment_count")
    .reset_index()
)

def recommend_from_cluster(user_id, top_n=None):
    user_cluster = clustered_users_df.loc[clustered_users_df["user"] == user_id, "cluster"]
    if user_cluster.empty:
        return []

    cluster = int(user_cluster.iloc[0])
    seen_courses = set(test_users_df.loc[test_users_df["user"] == user_id, "item"])
    popular_courses = (
        cluster_course_counts.loc[
            (cluster_course_counts["cluster"] == cluster)
            & (cluster_course_counts["cluster_enrollment_count"] > ENROLLMENT_THRESHOLD)
        ]
        .sort_values("cluster_enrollment_count", ascending=False)
    )
    recs = [
        course_id
        for course_id in popular_courses["item"].tolist()
        if course_id not in seen_courses
    ]
    return recs[:top_n] if top_n else recs

test_user_ids = sorted(test_users_df["user"].unique())
cluster_recommendations = {user_id: recommend_from_cluster(user_id) for user_id in test_user_ids}

avg_recommendations = np.mean([len(courses) for courses in cluster_recommendations.values()])
print(f"Average recommendations per test user: {avg_recommendations:.3f}")

Average recommendations per test user: 87.511


In [6]:
from collections import Counter

all_cluster_recs = [course for courses in cluster_recommendations.values() for course in courses]
top_cluster_recs = (
    pd.DataFrame(Counter(all_cluster_recs).most_common(10), columns=["COURSE_ID", "recommendation_count"])
    .merge(load_csv("course_genre.csv")[["COURSE_ID", "TITLE"]], on="COURSE_ID", how="left")
)

display(top_cluster_recs)

,COURSE_ID,recommendation_count,TITLE
0,COM001EN,993,scalable web applications on kubernetes
1,ML0122ENv3,992,accelerating deep learning with gpus
2,PHPM002EN,991,php web application on a lamp stack
3,CB0101EN,990,build your own chatbots
4,ML0120ENv3,990,deep learning with tensorflow
5,SW0201EN,987,how to build watson ai and swift apis and make...
6,SECM03EN,987,apply end to end security to a cloud application
7,IT0101EN,986,building robots with tjbot
8,EE0101EN,985,modernizing java ee applications
9,DAI101EN,984,data ai essentials


## Interpretation

Clustering is a middle-ground content-based method. It is broader than strict course similarity but more group-personalized than pure genre-vector matching.